# ML-04 — Search Intelligence Data Contract (Lane 2: Refresh / Content Opportunity Scoring)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meer140/flyrank-ml-internship-project/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook details the **Data Contract** for **Lane 2 (Refresh / Content Opportunity Scoring)**. It defines the unit of analysis, table dependencies, observation & decision windows, field classifications, verification queries, feature engineering rules, and a deliberate feature-leakage experiment on real warehouse data.

## 1. Unit of analysis + time window

### Plain Words Contract Answers (5 Answers):
1. **Unit of Analysis / Grain:** One pseudonymized content item (`content_hash_id` per `client_hash_id` in warehouse release, or `content_id` per `client_id` in starter CSV) evaluated over a candidate time window.
2. **Tables Used:** `fact_content_daily_performance` (partitioned by month, e.g. `month=2026-03`), `dim_content`, `dim_clients`, and `fact_content_query_90d` (with local starter CSV fallback).
3. **Time Window:** Mid-panel snapshot month (`2026-03-01` to `2026-03-31`), observing 30-day prior signals (`prev_30d`) to predict 30-day outcome signals (`last_30d`).
4. **Target (Label / Proxy):** Binary refresh/decline target `is_declining` (`imp_last30 < 0.8 * imp_prev30` for pages with `imp_prev30 >= 50`).
5. **Deliberately Excluded:** `trend_direction`, `trend_pct`, future outcome period performance metrics (`imp_last30`, `clk_last30`), and raw string IDs (`client_hash_id`, `content_hash_id`) as direct model features.

In [1]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Environment & Token Setup
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass

con = duckdb.connect()
use_hf = False

if HF_TOKEN:
    try:
        con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
        con.sql("SELECT COUNT(*) FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')").fetchone()
        use_hf = True
        print("Successfully connected to Hugging Face warehouse release!")
    except Exception as e:
        print(f"HF Token present but warehouse access check failed ({e}). Using local starter fallback.")
else:
    print("No HF_TOKEN detected in environment. Using local starter dataset fallback.")

if use_hf:
    REL = 'hf://datasets/FlyRank/internship-warehouse'
    TABLES = {
        'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
        'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
        'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
        'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
        'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
    }
else:
    possible_paths = [
        'data/raw/content_refresh_anonymized.csv',
        '../data/raw/content_refresh_anonymized.csv',
        '../../data/raw/content_refresh_anonymized.csv'
    ]
    local_path = None
    for p in possible_paths:
        if os.path.exists(p):
            local_path = p
            break
    if not local_path:
        raise FileNotFoundError("Could not locate data/raw/content_refresh_anonymized.csv")
    TABLES = {'starter_csv': f"'{local_path}'"}
    print(f"Local dataset verified at: {local_path}")


No HF_TOKEN detected in environment. Using local starter dataset fallback.
Local dataset verified at: ../../data/raw/content_refresh_anonymized.csv


## 2. Fields: feature / label / context / excluded

### Field Classification & Rationale:
- **Feature (Safe to model):** `imp_prev30`, `clk_prev30`, `pos_prev30`, `ctr_prev30`, `has_ga4_history` — knowable strictly BEFORE the decision moment.
- **Label / Proxy (Target):** `is_declining` (`imp_last30 < 0.8 * imp_prev30`) — binary outcome flag measuring whether organic traffic decayed >20% in the subsequent window.
- **Context (IDs / Splitting):** `client_hash_id` / `client_id`, `content_hash_id` / `content_id`, `content_type` — used for grouping, joining, and client-holdout splits.
- **Excluded (Leakage / Invalid):** `trend_direction`, `trend_pct`, `imp_last30`, `clk_last30` — future outcome period metrics that leak the target.

In [2]:
# Data Contract Field Classification Schema
schema_buckets = {
    "Feature": [
        "imp_prev30 (Historical search impressions in preceding 30 days)",
        "clk_prev30 (Historical organic clicks in preceding 30 days)",
        "pos_prev30 (Average search engine result position in preceding 30 days)",
        "ctr_prev30 (Historical click-through rate over preceding 30 days)",
        "has_ga4_history (Boolean flag indicating GA4 analytics availability)"
    ],
    "Label": [
        "is_declining (imp_last30 < 0.8 * imp_prev30: >20% impression drop in outcome window)"
    ],
    "Context": [
        "client_hash_id / client_id (Pseudonymized client identifier for grouping & splits)",
        "content_hash_id / content_id (Pseudonymized page identifier for joins)",
        "content_type (Content format category for stratified evaluation)"
    ],
    "Excluded": [
        "trend_direction (Derived directly from outcome window trend — LEAKAGE)",
        "trend_pct (Percentage metric computed across outcome period — LEAKAGE)",
        "imp_last30 / clk_last30 (Raw outcome window performance — LEAKAGE)"
    ]
}

print("=== DATA CONTRACT FIELD BUCKETS ===")
for bucket, fields in schema_buckets.items():
    print(f"\n{bucket.upper()}:")
    for f in fields:
        print(f"  - {f}")


=== DATA CONTRACT FIELD BUCKETS ===

FEATURE:
  - imp_prev30 (Historical search impressions in preceding 30 days)
  - clk_prev30 (Historical organic clicks in preceding 30 days)
  - pos_prev30 (Average search engine result position in preceding 30 days)
  - ctr_prev30 (Historical click-through rate over preceding 30 days)
  - has_ga4_history (Boolean flag indicating GA4 analytics availability)

LABEL:
  - is_declining (imp_last30 < 0.8 * imp_prev30: >20% impression drop in outcome window)

CONTEXT:
  - client_hash_id / client_id (Pseudonymized client identifier for grouping & splits)
  - content_hash_id / content_id (Pseudonymized page identifier for joins)
  - content_type (Content format category for stratified evaluation)

EXCLUDED:
  - trend_direction (Derived directly from outcome window trend — LEAKAGE)
  - trend_pct (Percentage metric computed across outcome period — LEAKAGE)
  - imp_last30 / clk_last30 (Raw outcome window performance — LEAKAGE)


## 3. Verify it with queries (grain, counts, missing values, windows)

Below we execute exactly **three verification queries** to prove key facts about our slice of the warehouse, followed by the **five-feature frame** construction and the **deliberate feature-leakage trap** experiment.

In [3]:
# Query 1: Probe Grain Uniqueness
if use_hf:
    grain_query = f"""
        SELECT client_hash_id, content_hash_id, COUNT(*) AS cnt
        FROM {TABLES['fact_daily']}
        WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
        GROUP BY 1, 2
        HAVING cnt > 31
        LIMIT 5
    """
else:
    grain_query = f"""
        SELECT client_id, content_id, COUNT(*) AS cnt
        FROM {TABLES['starter_csv']}
        GROUP BY 1, 2
        HAVING cnt > 1
        LIMIT 5
    """

grain_check = con.sql(grain_query).df()
print("=== QUERY 1: GRAIN VERIFICATION PROBE ===")
print(f"Duplicate grain rows returned (0 expected): {len(grain_check)}")
if len(grain_check) == 0:
    print("SUCCESS: Grain holds! One row uniquely represents one content item per observation window.")
else:
    print(grain_check)


=== QUERY 1: GRAIN VERIFICATION PROBE ===
Duplicate grain rows returned (0 expected): 0
SUCCESS: Grain holds! One row uniquely represents one content item per observation window.


In [4]:
# Query 2: Slice Row Count and Date Span / Range
if use_hf:
    count_query = f"""
        SELECT 
            COUNT(*) AS total_rows,
            COUNT(DISTINCT content_hash_id) AS distinct_content_items,
            COUNT(DISTINCT client_hash_id) AS distinct_clients,
            MIN(report_date) AS min_report_date,
            MAX(report_date) AS max_report_date
        FROM {TABLES['fact_daily']}
    """
else:
    count_query = f"""
        SELECT 
            COUNT(*) AS total_rows,
            COUNT(DISTINCT content_id) AS distinct_content_items,
            COUNT(DISTINCT client_id) AS distinct_clients,
            MIN(content_age_days) AS min_content_age_days,
            MAX(content_age_days) AS max_content_age_days
        FROM {TABLES['starter_csv']}
    """

count_df = con.sql(count_query).df()
print("=== QUERY 2: SLICE ROW COUNT AND DATE SPAN ===")
print(count_df.to_string(index=False))


=== QUERY 2: SLICE ROW COUNT AND DATE SPAN ===
 total_rows  distinct_content_items  distinct_clients  min_content_age_days  max_content_age_days
      30000                   30000                32                    90                   564


In [5]:
# Query 3: Availability Filter Check using IS TRUE
if use_hf:
    avail_query = f"""
        SELECT 
            COUNT(*) AS total_rows,
            SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
            SUM(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS both_available_rows,
            ROUND(100.0 * SUM(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_surviving
        FROM {TABLES['fact_daily']}
    """
else:
    avail_query = f"""
        SELECT 
            COUNT(*) AS total_rows,
            SUM(CASE WHEN (impressions_90d > 0) IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
            SUM(CASE WHEN (sessions_90d > 0) IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
            SUM(CASE WHEN (impressions_90d > 0) IS TRUE AND (sessions_90d > 0) IS TRUE THEN 1 ELSE 0 END) AS both_available_rows,
            ROUND(100.0 * SUM(CASE WHEN (impressions_90d > 0) IS TRUE AND (sessions_90d > 0) IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_surviving
        FROM {TABLES['starter_csv']}
    """

avail_df = con.sql(avail_query).df()
print("=== QUERY 3: AVAILABILITY FILTER CHECK (IS TRUE) ===")
print(avail_df.to_string(index=False))


=== QUERY 3: AVAILABILITY FILTER CHECK (IS TRUE) ===
 total_rows  gsc_available_rows  ga4_available_rows  both_available_rows  pct_surviving
      30000             30000.0             30000.0              30000.0          100.0


In [6]:
# Feature Frame Construction (5 Honest Features) + Availability Notes
if use_hf:
    ff_query = f"""
        SELECT 
            content_hash_id,
            client_hash_id,
            SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clk_prev30,
            AVG(CASE WHEN report_date <= '2026-03-15' THEN gsc_avg_position ELSE NULL END) AS pos_prev30,
            CASE WHEN SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) > 0 
                 THEN SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_clicks ELSE 0 END) / SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) 
                 ELSE 0 END AS ctr_prev30,
            MAX(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS has_ga4_history,
            SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_last30
        FROM {TABLES['fact_daily']}
        GROUP BY 1, 2
        HAVING imp_prev30 >= 50
    """
    df_ff = con.sql(ff_query).df()
    df_ff['is_declining'] = (df_ff['imp_last30'] < 0.8 * df_ff['imp_prev30']).astype(int)
else:
    df_raw = con.sql(f"SELECT * FROM {TABLES['starter_csv']}").df()
    df_ff = pd.DataFrame()
    df_ff['content_id'] = df_raw['content_id']
    df_ff['client_id'] = df_raw['client_id']
    df_ff['imp_prev30'] = df_raw['impressions_prev_30d']
    df_ff['clk_prev30'] = df_raw['clicks_prev_30d']
    df_ff['pos_prev30'] = df_raw['avg_position']
    df_ff['ctr_prev30'] = df_raw['clicks_prev_30d'] / np.maximum(df_raw['impressions_prev_30d'], 1)
    df_ff['has_ga4_history'] = (df_raw['sessions_90d'] > 0).astype(int)
    df_ff['imp_last30'] = df_raw['impressions_last_30d']
    df_ff['is_declining'] = (df_raw['trend_direction'] == 'down').astype(int)

df_ff['pos_prev30'] = df_ff['pos_prev30'].fillna(df_ff['pos_prev30'].median())

print(f"Feature Frame constructed successfully with {len(df_ff):,} content items.")
print("\n=== FIVE-FEATURE FRAME AVAILABILITY NOTES ===")
feature_notes = {
    "imp_prev30": "knowable at the decision moment because it measures historical search impressions prior to the prediction cutoff date.",
    "clk_prev30": "knowable at the decision moment because organic clicks are logged strictly prior to the evaluation point.",
    "pos_prev30": "knowable at the decision moment because SERP position ranks are recorded in historical search console logs.",
    "ctr_prev30": "knowable at the decision moment because it is a historical ratio calculated from past clicks and impressions.",
    "has_ga4_history": "knowable at the decision moment because web analytics tracking installation status is established prior to scoring."
}

for feat, note in feature_notes.items():
    print(f"• {feat}: {note}")

df_ff.head()


Feature Frame constructed successfully with 30,000 content items.

=== FIVE-FEATURE FRAME AVAILABILITY NOTES ===
• imp_prev30: knowable at the decision moment because it measures historical search impressions prior to the prediction cutoff date.
• clk_prev30: knowable at the decision moment because organic clicks are logged strictly prior to the evaluation point.
• pos_prev30: knowable at the decision moment because SERP position ranks are recorded in historical search console logs.
• ctr_prev30: knowable at the decision moment because it is a historical ratio calculated from past clicks and impressions.
• has_ga4_history: knowable at the decision moment because web analytics tracking installation status is established prior to scoring.


,content_id,client_id,imp_prev30,clk_prev30,pos_prev30,ctr_prev30,has_ga4_history,imp_last30,is_declining
0,content_304f48230142,client_f369cb89fc,987,13,10.6,0.013171,1,578,1
1,content_a1fb4e703a9e,client_4e07408562,5915,1,20.3,0.000169,1,2501,1
2,content_9aa793d4d895,client_7f2253d7e2,6089,3,36.5,0.000493,1,2382,1
3,content_331d6c4de07b,client_19581e27de,4206,17,6.2,0.004042,1,3626,0
4,content_d99b7a2d90ca,client_3fdba35f04,6452,2,44.0,0.000310,1,4211,1


In [7]:
# The Trap: Deliberate Feature Leakage Experiment
honest_features = ['imp_prev30', 'clk_prev30', 'pos_prev30', 'ctr_prev30', 'has_ga4_history']
X_honest = df_ff[honest_features]
y = df_ff['is_declining']

X_tr_h, X_te_h, y_tr, y_te = train_test_split(X_honest, y, test_size=0.25, random_state=42, stratify=y)

# 1. Train Honest Model
rf_honest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_honest.fit(X_tr_h, y_tr)
y_pred_honest = rf_honest.predict(X_te_h)
acc_honest = accuracy_score(y_te, y_pred_honest)

print("=== 1. HONEST MODEL EVALUATION (No Leakage) ===")
print(f"Honest Accuracy: {acc_honest:.4f}")
print(classification_report(y_te, y_pred_honest, digits=3))

# 2. Inject ONE Label-Derived Column On Purpose (The Trap!)
df_ff['leaked_imp_last30'] = df_ff['imp_last30']
leaked_features = honest_features + ['leaked_imp_last30']

X_leaked = df_ff[leaked_features]
X_tr_l, X_te_l, _, _ = train_test_split(X_leaked, y, test_size=0.25, random_state=42, stratify=y)

rf_leaked = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_leaked.fit(X_tr_l, y_tr)
y_pred_leaked = rf_leaked.predict(X_te_l)
acc_leaked = accuracy_score(y_te, y_pred_leaked)

print("\n=== 2. LEAKED MODEL EVALUATION (With Deliberately Injected Outcome Column `imp_last30`) ===")
print(f"TRAP TRIGGERED! Leaked Accuracy: {acc_leaked:.4f} (Artificially Perfect Score!)")
print(classification_report(y_te, y_pred_leaked, digits=3))

# 3. Cleanup & Honest Lesson
df_ff.drop(columns=['leaked_imp_last30'], inplace=True)
print("\n=== 3. CLEANUP & LEAKAGE LESSON ===")
print(f"Leaked feature removed from feature frame. Honest model accuracy maintained at {acc_honest:.4f}.")
print("Lesson Learned: Injecting outcome period data causes artificial high performance in training but complete model failure when deployed on unobserved live data.")


=== 1. HONEST MODEL EVALUATION (No Leakage) ===
Honest Accuracy: 0.6239
              precision    recall  f1-score   support

           0      0.597     0.550     0.572      3435
           1      0.643     0.687     0.664      4065

    accuracy                          0.624      7500
   macro avg      0.620     0.618     0.618      7500
weighted avg      0.622     0.624     0.622      7500




=== 2. LEAKED MODEL EVALUATION (With Deliberately Injected Outcome Column `imp_last30`) ===
TRAP TRIGGERED! Leaked Accuracy: 0.9779 (Artificially Perfect Score!)
              precision    recall  f1-score   support

           0      0.973     0.979     0.976      3435
           1      0.982     0.977     0.980      4065

    accuracy                          0.978      7500
   macro avg      0.978     0.978     0.978      7500
weighted avg      0.978     0.978     0.978      7500


=== 3. CLEANUP & LEAKAGE LESSON ===
Leaked feature removed from feature frame. Honest model accuracy maintained at 0.6239.
Lesson Learned: Injecting outcome period data causes artificial high performance in training but complete model failure when deployed on unobserved live data.


## 4. Data limits

### Named Limitation of Your Slice:
**Unbalanced History Depth and GA4 Analytics Cold-Starts Across Clients:**
Client history start dates vary substantially across the portfolio (`dim_clients.gsc_data_start` vs `dim_clients.ga4_data_start`). In early performance partitions, GA4 analytics columns are zero-filled with `ga4_data_available = FALSE`. A naive model treating zero-filled GA4 metrics as zero user engagement injects non-random category missingness; cross-client models must explicitly condition on `ga4_data_available IS TRUE` or train on client-aligned history windows.

In [8]:
# Data Limitation Probe: Empirical distribution of history start & GA4 tracking
if use_hf:
    limit_query = f"""
        SELECT 
            access_profile,
            COUNT(*) AS client_count,
            MIN(gsc_data_start) AS earliest_gsc_start,
            MAX(gsc_data_start) AS latest_gsc_start,
            SUM(CASE WHEN ga4_data_start IS NOT NULL THEN 1 ELSE 0 END) AS clients_with_ga4
        FROM {TABLES['dim_clients']}
        GROUP BY 1
    """
    limit_df = con.sql(limit_query).df()
else:
    limit_query = f"""
        SELECT 
            content_type,
            COUNT(*) AS page_count,
            SUM(CASE WHEN sessions_90d = 0 THEN 1 ELSE 0 END) AS zero_session_pages,
            ROUND(100.0 * SUM(CASE WHEN sessions_90d = 0 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_zero_sessions
        FROM {TABLES['starter_csv']}
        GROUP BY 1
    """
    limit_df = con.sql(limit_query).df()

print("=== DATA LIMITATION EVIDENCE SUMMARY ===")
print(limit_df.to_string(index=False))


=== DATA LIMITATION EVIDENCE SUMMARY ===
      content_type  page_count  zero_session_pages  pct_zero_sessions
comparison article         697                 0.0                0.0
   keyword article       27207                 0.0                0.0
    feedly article        2096                 0.0                0.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.